# DAC26 ROSA: simulated EDP reproduction
This notebook consumes one completed `reproduce` run. It never reads the committed reference CSVs as experiment results and intentionally excludes accuracy/noise evaluation.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
run_dir = Path(os.environ['OPTICALLOOP_RUN_DIR']).resolve()
metadata = json.loads((run_dir / 'run.json').read_text())
artifacts = run_dir / 'artifacts'
raw = pd.read_csv(artifacts / 'layer_results.csv')
aggregates = pd.read_csv(artifacts / 'network_architecture_metrics.csv')
checks = pd.read_csv(artifacts / 'validation.csv')
headline = json.loads((artifacts / 'headline.json').read_text())
metadata['provenance'], metadata['tier'], metadata['successful_jobs'], metadata['expected_jobs']

## Coverage and simulator invariants

In [ ]:
coverage = raw.groupby(['network', 'variant', 'architecture']).size().rename('simulated_layers').reset_index()
display(coverage)
display(checks)

## EDP reconstructed from simulated energy and latency

In [ ]:
raw['recomputed_edp_j_s'] = raw.energy_j * raw.latency_s
assert ((raw.edp_j_s - raw.recomputed_edp_j_s).abs() <= raw.edp_j_s.abs().clip(lower=1e-30) * 1e-9).all()
display(aggregates)

## Full-sweep architecture and OSA comparison
Headline metrics are available only for the full tier. Paper deltas remain visible where the publication omits λ or raw ODE inputs.

In [ ]:
display(headline)
if headline.get('available'):
    chart = aggregates.groupby(['architecture', 'variant']).edp_j_s.apply(lambda s: (s.prod()) ** (1 / len(s))).unstack()
    ax = chart.plot.bar(figsize=(11, 5), logy=True, ylabel='Geometric-mean EDP (J·s)')
    ax.set_title('DAC26 six-workload architecture sweep')
    plt.tight_layout()
    plt.savefig(artifacts / 'architecture_edp.png', dpi=160)
    plt.show()

## Conclusion
The authoritative result is `artifacts/REPORT.md`. `PASS_WITH_PAPER_GAPS` means all simulated invariants and reference checks pass while one or more publication-only claims cannot be independently reconstructed from disclosed inputs.